# Scorecard

In [360]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

from typing import List, Dict
from sklearn.model_selection import train_test_split

In [361]:
dataset_path = "../data/raw/cs-training.csv"

df = pd.read_csv(dataset_path)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns = ["Unnamed: 0"])

print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
df.head()

Total Rows: 150000, Total Columns: 11


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [362]:
df = df[df["age"] > 0]
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")

Total Rows: 149999, Total Columns: 11


In [363]:
X = df.drop(columns = ["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

print(f"X -> Total Rows: {X.shape[0]}, Total Coulmns: {X.shape[1]}")
print(f"y -> Total Rows: {y.shape[0]}")

X -> Total Rows: 149999, Total Coulmns: 10
y -> Total Rows: 149999


In [364]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify = y, random_state = 42
)

print(y_train.mean())
print(y_test.mean())

0.0668361496546636
0.06685333333333333


#### Binning and WoE Calculation

In [365]:
def bin_features(series: pd.Series, bins: List, labels: List[str], right: bool = False, missing_label = "Missing"):
    group = pd.cut(series, bins = bins, labels = labels, right = right, include_lowest = True)

    group = group.cat.add_categories(missing_label).fillna(missing_label)

    return group.cat.remove_unused_categories()

In [366]:
def calculate_woe(df: pd.DataFrame, feature: str, target: str = "SeriousDlqin2yrs", epilson: float = 1e-6) -> tuple[pd.DataFrame, float]:
    total_goods = (df[target] == 0).sum()
    total_bads = (df[target] == 1).sum()

    summary = df.groupby(feature, observed = False)[target].agg(
        total = "count",
        bads = "sum"
    ).reset_index()

    summary["goods"] = summary["total"] - summary["bads"]
    summary["bad_rate"] = (summary["bads"] / summary["total"] * 100).round(2)

    summary["percentage_good"] = summary["goods"] / total_goods
    summary["percentage_bad"] = summary["bads"] / total_bads

    summary["percentage_good"] = summary["percentage_good"].replace(0, epilson)
    summary["percentage_bad"] = summary["percentage_bad"].replace(0, epilson)

    summary["WoE"] = np.log(summary["percentage_good"] / summary["percentage_bad"])
    summary["IV"] = (summary["percentage_good"] - summary["percentage_bad"]) * summary["WoE"]

    total_iv = summary["IV"].sum()
    
    return summary, total_iv

In [367]:
BINNING_CONFIG = {
    "RevolvingUtilizationOfUnsecuredLines": {
        "bins": [-np.inf, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, np.inf],
        "labels": ["<10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%", ">100%"]
    },
    
    "age": {
        "bins": [-np.inf, 30, 40, 50, 60, 70, np.inf],
        "labels": ["<30", "30-39", "40-49", "50-59", "60-69", "70+"]
    },
    
    "NumberOfTime30-59DaysPastDueNotWorse": {
        "bins": [-np.inf, 1, 2, np.inf],
        "labels": ["0", "1", "2+"]
    },
    
    "DebtRatio": {
        "bins": [-np.inf, 0.2, 0.4, 0.6, 1.0, 5.0, np.inf],
        "labels": ["<20%", "20-40%", "40-60%", "60-100%", "1-5x", ">5x"]
    },
    
    "MonthlyIncome": {
        "bins": [-np.inf, 2500, 5000, 7500, 10000, 15000, np.inf],
        "labels": ["<2.5k", "2.5k-5k", "5k-7.5k", "7.5k-10k", "10k-15k", ">15k"]
    },
    
    "NumberOfOpenCreditLinesAndLoans": {
        "bins": [-np.inf, 3, 6, 10, 15, np.inf],
        "labels": ["0-2", "3-5", "6-9", "10-14", "15+"]
    },
    
    "NumberOfTimes90DaysLate": {
        "bins": [-np.inf, 1, 2, np.inf],
        "labels": ["0", "1", "2+"]
    },
    
    "NumberRealEstateLoansOrLines": {
        "bins": [-np.inf, 1, 2, 3, np.inf],
        "labels": ["0", "1", "2", "3+"]
    },
    
    "NumberOfTime60-89DaysPastDueNotWorse": {
        "bins": [-np.inf, 1, 2, np.inf],
        "labels": ["0", "1", "2+"]
    },
    
    "NumberOfDependents": {
        "bins": [-np.inf, 1, 2, 3, np.inf],
        "labels": ["0", "1", "2", "3+"]
    }
}

In [368]:
def woe_mapping(config: Dict = BINNING_CONFIG) -> tuple[Dict, Dict]:
    woe_maps = {}
    IV_map = {}
    for feature, config in BINNING_CONFIG.items():
        bins = config["bins"]
        labels = config["labels"]
        
        group = bin_features(X_train[feature], bins, labels)
        group_name = f"{feature}_group"
        X_train[group_name] = group

        df_concat = pd.concat([X_train, y_train], axis = 1)

        summary, total_iv = calculate_woe(df_concat, group_name)

        woe_maps[group_name] = {
            row[group_name]: {
                "WoE": row["WoE"],
                "IV": row["IV"]
            }for _, row in summary.iterrows()
        }

        IV_map[group_name] = total_iv
    return woe_maps, IV_map


In [369]:
woe_maps, IV_map = woe_mapping()

IV_map

{'RevolvingUtilizationOfUnsecuredLines_group': np.float64(1.135659903577781),
 'age_group': np.float64(0.2313969900530205),
 'NumberOfTime30-59DaysPastDueNotWorse_group': np.float64(0.7245518903290449),
 'DebtRatio_group': np.float64(0.06702460502815792),
 'MonthlyIncome_group': np.float64(0.07163253619739691),
 'NumberOfOpenCreditLinesAndLoans_group': np.float64(0.08134062680351581),
 'NumberOfTimes90DaysLate_group': np.float64(0.8816049260728693),
 'NumberRealEstateLoansOrLines_group': np.float64(0.05616584632431472),
 'NumberOfTime60-89DaysPastDueNotWorse_group': np.float64(0.5990007010474803),
 'NumberOfDependents_group': np.float64(0.035308274158114196)}

### Transformation

In [370]:
def transform_woe(df, binning_config, woe_map):
    df_woe = pd.DataFrame(index = df.index)

    for feature, config in binning_config.items():
        bins = config["bins"]
        labels = config["labels"]

        group = bin_features(df[feature], bins, labels)
        group_name = f"{feature}_group"

        feature_woe_lookup = {
            k: v["WoE"] for k, v in woe_map[group_name].items()
        }

        df_woe[f"{feature}_group"] = (group.astype(str).map(feature_woe_lookup)).astype(float)

    return df_woe

In [371]:
X_train_woe = transform_woe(X_train, BINNING_CONFIG, woe_maps)
print(f"Number of NaN values:\n{X_train_woe.isnull().sum()}")
X_train_woe.head(3)

Number of NaN values:
RevolvingUtilizationOfUnsecuredLines_group    0
age_group                                     0
NumberOfTime30-59DaysPastDueNotWorse_group    0
DebtRatio_group                               0
MonthlyIncome_group                           0
NumberOfOpenCreditLinesAndLoans_group         0
NumberOfTimes90DaysLate_group                 0
NumberRealEstateLoansOrLines_group            0
NumberOfTime60-89DaysPastDueNotWorse_group    0
NumberOfDependents_group                      0
dtype: int64


,RevolvingUtilizationOfUnsecuredLines_group,age_group,NumberOfTime30-59DaysPastDueNotWorse_group,DebtRatio_group,MonthlyIncome_group,NumberOfOpenCreditLinesAndLoans_group,NumberOfTimes90DaysLate_group,NumberRealEstateLoansOrLines_group,NumberOfTime60-89DaysPastDueNotWorse_group,NumberOfDependents_group
82345,1.335872,0.019967,0.532956,0.170463,0.019974,0.136895,0.39311,0.188866,0.289341,-0.106676
53120,-0.420523,-0.238865,0.532956,-0.156802,-0.288021,-0.001019,0.39311,0.259024,-2.744277,-0.106676
33898,1.335872,-0.432068,0.532956,0.099317,-0.288021,0.200614,0.39311,-0.234781,0.289341,0.138520


In [372]:
X_test_woe = transform_woe(X_test, BINNING_CONFIG, woe_maps)
X_test_woe.head(3)

,RevolvingUtilizationOfUnsecuredLines_group,age_group,NumberOfTime30-59DaysPastDueNotWorse_group,DebtRatio_group,MonthlyIncome_group,NumberOfOpenCreditLinesAndLoans_group,NumberOfTimes90DaysLate_group,NumberRealEstateLoansOrLines_group,NumberOfTime60-89DaysPastDueNotWorse_group,NumberOfDependents_group
22984,-0.903891,-0.592529,0.532956,0.221570,-0.299549,-0.001019,0.39311,-0.234781,0.289341,0.138520
61308,1.335872,0.019967,0.532956,0.099317,0.495323,0.136895,0.39311,-0.234781,0.289341,-0.210153
24228,1.335872,1.093926,0.532956,0.099317,-0.288021,0.136895,0.39311,-0.234781,0.289341,0.138520


### Training and Testing

In [373]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [374]:
y_train_good = 1 - y_train
y_test_good = 1 - y_test

In [375]:
clf = LogisticRegression(penalty = None, solver = "lbfgs", max_iter = 1000).fit(X_train_woe, y_train_good)

/media/divyanshu/MyStorage1/Learning/01_Credit_Fraud/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [376]:
coef_df = pd.DataFrame({
    "Feature": X_train_woe.columns,
    "Coefficient": clf.coef_.ravel(),
    "Odds Ratio": np.exp(clf.coef_.ravel())
})

coef_df = coef_df.sort_values(by = "Coefficient", key = abs, ascending = False)

print(f"Intercept: {clf.intercept_[0]:.4f}")
display(coef_df)

Intercept: 2.6077


,Feature,Coefficient,Odds Ratio
3,DebtRatio_group,0.865419,2.376001
0,RevolvingUtilizationOfUnsecuredLines_group,0.613875,1.847577
7,NumberRealEstateLoansOrLines_group,0.584271,1.793682
6,NumberOfTimes90DaysLate_group,0.532678,1.703487
2,NumberOfTime30-59DaysPastDueNotWorse_group,0.508614,1.662985
8,NumberOfTime60-89DaysPastDueNotWorse_group,0.381684,1.464749
1,age_group,0.378334,1.459851
9,NumberOfDependents_group,0.280178,1.323365
5,NumberOfOpenCreditLinesAndLoans_group,0.063302,1.065348
4,MonthlyIncome_group,0.059782,1.061605


In [377]:
y_pred_prob = clf.predict_proba(X_test_woe)[:, 1]

In [378]:
auc_roc = roc_auc_score(y_test_good, y_pred_prob)

gini_coef = 2 * auc_roc - 1

print(f"AUC-ROC Score: {auc_roc}")
print(f"Gini Coefficient: {gini_coef}")

AUC-ROC Score: 0.8589953217722011
Gini Coefficient: 0.7179906435444021


### Points Table

In [379]:
print(woe_maps)

{'RevolvingUtilizationOfUnsecuredLines_group': {'<10%': {'WoE': 1.3358717849892154, 'IV': 0.4457577271099729}, '10-30%': {'WoE': 0.8327626751578888, 'IV': 0.09212382488183982}, '30-50%': {'WoE': 0.15568923584859748, 'IV': 0.002410355817098539}, '50-70%': {'WoE': -0.42052349714632903, 'IV': 0.0160319641296484}, '70-90%': {'WoE': -0.9038913332009738, 'IV': 0.08011207131228372}, '90-100%': {'WoE': -1.2070981219796313, 'IV': 0.2711926903191288}, '>100%': {'WoE': -2.104188312193452, 'IV': 0.22803127000780876}}, 'age_group': {'<30': {'WoE': -0.5925291859951224, 'IV': 0.02667005593081}, '30-39': {'WoE': -0.4320680415883385, 'IV': 0.034856976376149255}, '40-49': {'WoE': -0.23886494610395187, 'IV': 0.01449238684013315}, '50-59': {'WoE': 0.019966638543334386, 'IV': 9.323769163237807e-05}, '60-69': {'WoE': 0.618112995538053, 'IV': 0.056468276706462144}, '70+': {'WoE': 1.0939259891867048, 'IV': 0.09881605650783355}}, 'NumberOfTime30-59DaysPastDueNotWorse_group': {'0': {'WoE': 0.5329564062392569, '

In [380]:
def scorecard(df: pd.DataFrame, woe_map: Dict, target_score: int = 600, target_odds: int = 50, pdo: int = 20) -> tuple[int, pd.DataFrame]:
    factor = pdo / np.log(2)
    offset = target_score - factor * np.log(target_odds)

    base_points = round(offset + factor * clf.intercept_[0])

    coef_dict = dict(zip(X_train_woe.columns, clf.coef_[0]))
    scorecard_rows = []
    for feature, coef in coef_dict.items():
        for bin_label, metrics in woe_map[feature].items():
            woe = metrics["WoE"]
            points = round(factor * coef * woe)

            scorecard_rows.append({
                "Feature": feature,
                "Bin": bin_label,
                "WoE": woe,
                "Points": points
            })

    scorecard = pd.DataFrame(scorecard_rows)


    return base_points, offset, factor, scorecard

In [381]:
base_points, offset, factor, scorecard = scorecard(X_test_woe, woe_maps)
print(f"Base Points: {base_points}, Offset: {offset}, Factor: {factor}")
print(scorecard.head(15))

Base Points: 562, Offset: 487.1228762045055, Factor: 28.85390081777927
                                       Feature      Bin       WoE  Points
0   RevolvingUtilizationOfUnsecuredLines_group     <10%  1.335872      24
1   RevolvingUtilizationOfUnsecuredLines_group   10-30%  0.832763      15
2   RevolvingUtilizationOfUnsecuredLines_group   30-50%  0.155689       3
3   RevolvingUtilizationOfUnsecuredLines_group   50-70% -0.420523      -7
4   RevolvingUtilizationOfUnsecuredLines_group   70-90% -0.903891     -16
5   RevolvingUtilizationOfUnsecuredLines_group  90-100% -1.207098     -21
6   RevolvingUtilizationOfUnsecuredLines_group    >100% -2.104188     -37
7                                    age_group      <30 -0.592529      -6
8                                    age_group    30-39 -0.432068      -5
9                                    age_group    40-49 -0.238865      -3
10                                   age_group    50-59  0.019967       0
11                                   age_

In [388]:
def credit_score(df, base_points, offset, factor, scorecard):
    logit = clf.decision_function(df)
    test_scores = pd.Series((np.round(offset + factor * logit)), index = df.index)

    return test_scores

In [389]:
scores = credit_score(X_test_woe, base_points, offset, factor, scorecard)
scores.describe()

count    37500.000000
mean       582.674320
std         33.979467
min        405.000000
25%        566.000000
50%        592.000000
75%        608.000000
max        629.000000
dtype: float64

### Good Vs Bad Score

In [392]:
from scipy.stats import ks_2samp

goods_scores = scores[y_test == 0]
bads_scores = scores[y_test == 1]


ks_stat, p_value = ks_2samp(goods_scores, bads_scores)

print(f"KS Statistic: {ks_stat*100:.2f}% (p-value: {p_value})")

KS Statistic: 56.45% (p-value: 0.0)
